In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# LongSplat 환경 세팅 (Colab A100)

> **[NVlabs/LongSplat](https://github.com/NVlabs/LongSplat)** — ICCV 2025  
> Colab A100 환경 기준 (CUDA 12.1 / PyTorch 2.5.1)

---

## ✅ 순서
1. GPU 확인
2. Git Clone
3. CUDA / GCC 세팅
4. PyTorch 및 패키지 설치
5. Submodule 빌드
6. 데이터셋 업로드
7. 학습 실행


## 0. GPU 확인

In [ ]:
!nvidia-smi

## 1. Git Clone

In [ ]:
!git clone --recursive https://github.com/NVlabs/LongSplat.git
%cd LongSplat

## 2. CUDA 12.1 PATH 설정 및 GCC 9 설치

In [ ]:
import os
os.environ["PATH"] = "/usr/local/cuda-12.1/bin:" + os.environ.get("PATH", "")
os.environ["LD_LIBRARY_PATH"] = "/usr/local/cuda-12.1/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")

In [ ]:
!gcc --version

In [ ]:
!apt-get install -y gcc-9 g++-9 -q
!update-alternatives --install /usr/bin/gcc gcc /usr/bin/gcc-9 9
!update-alternatives --install /usr/bin/g++ g++ /usr/bin/g++-9 9
!update-alternatives --set gcc /usr/bin/gcc-9
!update-alternatives --set g++ /usr/bin/g++-9
!gcc --version

## 3. PyTorch 2.5.1 + CUDA 12.1 설치

In [ ]:
!pip install torch==2.5.1 torchvision --index-url https://download.pytorch.org/whl/cu121 -q

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

## 4. pytorch3d & torch_scatter 설치 (pre-built wheel)

In [ ]:
!pip install --extra-index-url https://miropsota.github.io/torch_packages_builder \
    pytorch3d==0.7.8+pt2.5.1cu121 -q

In [ ]:
!pip install torch_scatter -f https://data.pyg.org/whl/torch-2.5.1+cu121.html -q

## 5. requirements.txt 설치

In [ ]:
!pip install -r requirements.txt -q

## 6. Submodule 빌드

In [ ]:
!pip install submodules/simple-knn --no-build-isolation -q

In [ ]:
!pip install submodules/diff-gaussian-rasterization --no-build-isolation -q

In [ ]:
# A100: Compute Capability 8.0
import os
os.environ["TORCH_CUDA_ARCH_LIST"] = "8.0"

%cd submodules/fused-ssim
!pip install -e . --no-build-isolation -q
%cd /content/LongSplat

## 7. (선택) RoPE CUDA 커널 컴파일

런타임 속도 향상을 위한 선택 사항입니다.

In [ ]:
%%bash
cd submodules/mast3r/dust3r/croco/models/curope/
python setup.py build_ext --inplace
cd /content/LongSplat

## 8. 데이터셋 준비

### Option A: Google Drive 마운트 후 복사

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.makedirs("data", exist_ok=True)

# 예시: Drive에 올려둔 커스텀 데이터셋 복사
# !cp -r /content/drive/MyDrive/YOUR_DATASET ./data/custom

### Option B: 파일 직접 업로드

In [ ]:
from google.colab import files
# uploaded = files.upload()  # 필요 시 주석 해제

## 9. 학습 실행

파라미터를 수정한 뒤 셀을 다시 실행하면 스크립트가 덮어씌워지고 바로 학습이 시작됩니다.

In [ ]:
import os, random, datetime, textwrap
from pathlib import Path

SCENES = ["grass_snow", "hydrant_snow", "pillar_snow", "road_snow", "sky_snow", "stair_snow",
          "grass_rain", "hydrant_rain", "pillar_rain", "road_rain", "sky_rain", "stair_rain"]

for scene in SCENES:

    # ── 파라미터 수정 후 셀 전체 실행 ──────────────────────────────────────────
    images       = scene
    input        = '/content/drive/MyDrive/Research/Dataset/free_outdoor_weathered'
    r            = 1
    pose_iter    = 100
    local_iter   = 200
    global_iter  = 500
    port         = random.randint(10000, 30000)
    timestamp    = datetime.datetime.now().strftime("%Y-%m-%d_%H:%M:%S")
    outputs   = f"/content/drive/MyDrive/Research/outputs/{scene}/{timestamp}"
    # ──────────────────────────────────────────────────────────────────────────

    script = textwrap.dedent(f"""
        #!/bin/bash
        set -e
        ulimit -n 4096
        python train.py --eval -s {input} -m {outputs} --port {port} \\
            --images {images} --mode custom -r {r} \\
            --pose_iteration {pose_iter} --local_iter {local_iter} --global_iter {global_iter}
        python render.py  -m {outputs}
        python metrics.py -m {outputs}
    """).strip()

    script_path = Path("scripts/train_custom.sh")
    script_path.write_text(script, encoding="utf-8")
    print(script_path.read_text())

    os.system(f"bash {script_path}")